In [126]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression 
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [127]:
df = pd.read_csv("Student_Performance_Prediction.csv")

col = list(df.columns)
samples = df.shape[0]
target = df.columns[-1]
features = df.drop(columns=[target])

print(col)
print(samples)
print(features.shape[1])
print(target)

['Study_Hours', 'Attendance_Percent', 'Practice_Tests', 'Sleep_Hours', 'Assignments_Completed', 'Screen_Time_Hours', 'Final_Score']
500
6
Final_Score


In [128]:
# correlation between each features
cor_mt = df.corr(method='pearson')
# print(cmat)

# correlation between feature & target
target_cr = cor_mt["Final_Score"]
# print(target_cr)

# correlation ranked
ranked = target_cr.abs().sort_values(ascending = False)
print(ranked)

Final_Score              1.000000
Practice_Tests           0.885623
Study_Hours              0.878533
Assignments_Completed    0.277262
Screen_Time_Hours        0.207730
Sleep_Hours              0.178447
Attendance_Percent       0.154457
Name: Final_Score, dtype: float64


In [129]:
print(cor_mt[cor_mt.abs() >= 0.70])

                       Study_Hours  Attendance_Percent  Practice_Tests  \
Study_Hours               1.000000                 NaN        0.886156   
Attendance_Percent             NaN                 1.0             NaN   
Practice_Tests            0.886156                 NaN        1.000000   
Sleep_Hours                    NaN                 NaN             NaN   
Assignments_Completed          NaN                 NaN             NaN   
Screen_Time_Hours              NaN                 NaN             NaN   
Final_Score               0.878533                 NaN        0.885623   

                       Sleep_Hours  Assignments_Completed  Screen_Time_Hours  \
Study_Hours                    NaN                    NaN                NaN   
Attendance_Percent             NaN                    NaN                NaN   
Practice_Tests                 NaN                    NaN                NaN   
Sleep_Hours                    1.0                    NaN                NaN   
Assignm

In [130]:
strong = target_cr[target_cr.abs() >= 0.40]
print(strong)

Study_Hours       0.878533
Practice_Tests    0.885623
Final_Score       1.000000
Name: Final_Score, dtype: float64


In [131]:
new_df = df[strong.index]

new_df.to_csv("modified_Student_Performance.csv")
print(new_df.head(5))

   Study_Hours  Practice_Tests  Final_Score
0          3.2             2.3        60.31
1          7.9             6.4        80.57
2          4.1             4.4        68.77
3          6.3             5.3        74.02
4          2.1             2.0        61.62


In [132]:
x1 = df.drop(columns = [target])
y1 = df[target]

xtrain1, xtest1, ytrain1, ytest1 = train_test_split(x1, y1, test_size = 0.2, random_state = 42)

model1 = LinearRegression()
model1.fit(xtrain1, ytrain1)

y_pred1 = model1.predict(xtest1)

mae1 = mean_absolute_error(ytest1, y_pred1)
mse1 = mean_squared_error(ytest1, y_pred1)
rmse1 = np.sqrt(mse1)
r2 = r2_score(ytest1, y_pred1)



x2 = df2.drop(columns = [target])
y2 = df2[target]

xtrain2, xtest2, ytrain2, ytest2 = train_test_split(x2, y2, test_size = 0.2, random_state = 42)

model2 = LinearRegression()
model2.fit(xtrain2, ytrain2)

y_pred2 = model2.predict(xtest2)

mae2 = mean_absolute_error(ytest2, y_pred2)
mse2 = mean_squared_error(ytest2, y_pred2)
rmse2 = np.sqrt(mse)
r_2 = r2_score(ytest2, y_pred2)

results = pd.DataFrame(
    {"Original": [mae1, mse1, rmse1, r2],
     "Modified": [mae2, mse2, rmse2, r_2]},
    index=["MAE", "MSE", "RMSE", "R2"]
)
print(results.round(4))


      Original  Modified
MAE     1.8174    2.9273
MSE     5.3229   13.6516
RMSE    2.3071    2.3071
R2      0.9248    0.8072


In [133]:
m1 = model1.coef_
c1 = model1.intercept_

m2 = model2.coef_
c2 = model2.intercept_


eq1 = f"{target} = " + " + ".join(f"({m:.3f} * {f})" for m, f in zip(m1, x1.columns)) + f" + {c1:.3f}"
eq2 = f"{target} = " + " + ".join(f"({m:.3f} * {f})" for m, f in zip(m2, x2.columns)) + f" + {c2:.3f}"

print("Slopes:", m1.round(3), " Intercept:", round(c1, 3))
print("Slopes", m2.round(3), " Intercept:", round(c2, 3))
print("\nOriginal equation:\n", eq1)
print("\nModified equation:\n", eq2)

Slopes: [ 1.759  0.093  1.821 -0.09   1.136 -0.616]  Intercept: 41.6
Slopes [1.518 2.201]  Intercept: 51.48

Original equation:
 Final_Score = (1.759 * Study_Hours) + (0.093 * Attendance_Percent) + (1.821 * Practice_Tests) + (-0.090 * Sleep_Hours) + (1.136 * Assignments_Completed) + (-0.616 * Screen_Time_Hours) + 41.600

Modified equation:
 Final_Score = (1.518 * Study_Hours) + (2.201 * Practice_Tests) + 51.480


In [134]:
#error = Actual - Predicted
error1 = ytest1.values - y_pred1      
error2 = ytest2.values - y_pred2

In [135]:
x = df.drop(columns = [target])
y = df[target]
e = []

for d in [2,3,4]:
    poly_F = PolynomialFeatures(degree = d, include_bias = False)
    x_poly = poly_F.fit_transform(x)

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

    model = LinearRegression()
    model.fit(x_train, y_train)

    print(d)
    y_pred_tr = model.predict(x_train)

    mse_tr = mean_squared_error(y_train, y_pred_tr)
    R2_tr = r2_score(y_train, y_pred_tr)

    print(mse_tr)
    print(R2_tr)

    y_pred = model.predict(x_test)

    mse = mean_squared_error(y_test, y_pred)
    R2 = r2_score(y_test, y_pred)

    print(mse)
    print(R2)

    e.append([d, mse_tr, mse, R2_tr, R2])

2
4.506131196009789
0.938668753933032
5.322854976253407
0.9248435687133907
3
4.506131196009789
0.938668753933032
5.322854976253407
0.9248435687133907
4
4.506131196009789
0.938668753933032
5.322854976253407
0.9248435687133907
